In [29]:
import pandas as pd

raw = pd.read_csv("Exam_Score_Prediction.csv")

In [30]:
raw_trunc = raw.drop(columns=["student_id", "age", "gender", "course", "internet_access", "sleep_hours", "exam_difficulty"])

raw_trunc.head()

,study_hours,class_attendance,sleep_quality,study_method,facility_rating,exam_score
0,2.78,92.9,poor,coaching,low,58.9
1,3.37,64.8,average,online videos,medium,54.8
2,7.88,76.8,poor,coaching,high,90.3
3,0.67,48.4,average,online videos,low,29.7
4,0.89,71.6,poor,coaching,low,43.7


In [31]:
y = raw_trunc["exam_score"]

X = raw_trunc.drop(columns=["exam_score"])

In [32]:
numeric_columns = X.select_dtypes(include=["number"]).columns.tolist()
categorical_columns = X.select_dtypes(exclude=["number"]).columns.tolist()



In [ ]:
import numpy as np

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [34]:
class IQRClipper(BaseEstimator, TransformerMixin):
    def __init__(self, k=1.5):
        self.k = k

    def fit(self, X, y):
        X = np.asarray(X, dtype=float)
        q1 = np.nanpercentile(X, 25, axis=0)
        q3 = np.nanpercentile(X, 75, axis=0)
        iqr = q3 - q1
        self.lower = q1 - self.k * iqr
        self.upper = q3 + self.k * iqr
        return self

    def transform(self, X):
        X = np.asarray(X, dtype=float)
        return np.clip(X, self.lower, self.upper)

In [35]:
num_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("clipper", IQRClipper(k=1.5)),
    ("scaler", StandardScaler())
])

cat_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numeric_columns),
        ("cat", cat_pipeline, categorical_columns)
    ]
)

model = Pipeline(steps=[
    ("preprocess", preprocess),
    ("regressor", Ridge(alpha=1.0))
])

In [37]:
from sklearn.model_selection import cross_validate

# KFold

In [38]:
from sklearn.model_selection import KFold

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scoring = {
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "r2": "r2"
}

cv_results = cross_validate(
    model,
    X,
    y,
    cv=kf,
    scoring=scoring,
    return_train_score=False,
    n_jobs=-1
)

mae_scores = -cv_results["test_mae"]
mse_scores = -cv_results["test_mse"]
rmse_scores = np.sqrt(mse_scores)
r2_scores = cv_results["test_r2"]

print("\nKFold (5) results:")
print(f"MAE  : {mae_scores.mean():.3f} ± {mae_scores.std():.3f}")
print(f"RMSE : {rmse_scores.mean():.3f} ± {rmse_scores.std():.3f}")
print(f"R^2  : {r2_scores.mean():.3f} ± {r2_scores.std():.3f}")


KFold (5) results:
MAE  : 8.110 ± 0.080
RMSE : 10.105 ± 0.092
R^2  : 0.714 ± 0.004


# ShuffleSplit

In [ ]:
from sklearn.model_selection import ShuffleSplit

ss = ShuffleSplit(
    n_splits=5,
    test_size=0.2,
    random_state=42
)

scoring = {
    "mae": "neg_mean_absolute_error",
    "mse": "neg_mean_squared_error",
    "r2": "r2"
}

cv_results_ss = cross_validate(
    model,
    X,
    y,
    cv=ss,
    scoring=scoring,
    n_jobs=-1
)

mae_scores_ss = -cv_results_ss["test_mae"]
rmse_scores_ss = np.sqrt(-cv_results_ss["test_mse"])
r2_scores_ss = cv_results_ss["test_r2"]

print("ShuffleSplit results:")
print(f"MAE  : {mae_scores_ss.mean():.3f} ± {mae_scores_ss.std():.3f}")
print(f"RMSE : {rmse_scores_ss.mean():.3f} ± {rmse_scores_ss.std():.3f}")
print(f"R²   : {r2_scores_ss.mean():.3f} ± {r2_scores_ss.std():.3f}")

ShuffleSplit results:
MAE  : 8.129 ± 0.085
RMSE : 10.139 ± 0.087
R²   : 0.712 ± 0.005
